# LoRA for Generative (Decoder-Only) Models

In [ ]:
# ========================================= 
# 1. Load base model and attach LoRA adapter 
# =========================================
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

# a beginner-friendly, modest-sized
# instruct model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)
base_model = (
    AutoModelForCausalLM
    .from_pretrained(model_name)
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
    ],
    # decoder-only models use
    # q_proj/v_proj/k_proj/o_proj
    # instead of DistilBERT's
    # q_lin/v_lin
)

model = get_peft_model(
    base_model, lora_config
)
model.print_trainable_parameters()

In [ ]:
# =========================================
# 2. Build and format the training data
# =========================================
from datasets import Dataset

# a small, concrete example dataset —
# replace with your real dataset
raw_dataset = Dataset.from_list([
    {
        "instruction": (
            "Summarize this review in one "
            "sentence: 'The pho was rich "
            "and flavorful, but we waited "
            "45 minutes just to order.'"
        ),
        "response": (
            "Great pho, but painfully "
            "slow service."
        ),
    },
    {
        "instruction": (
            "What star rating (1-5) fits "
            "this review: 'Cold food, "
            "rude staff, would not "
            "return.'"
        ),
        "response": "1 star.",
    },
    {
        "instruction": (
            "Extract the main complaint "
            "from this review: 'Loved "
            "the ambiance and cocktails, "
            "but the music was way too "
            "loud to hold a conversation.'"
        ),
        "response": "The music was too loud.",
    },
    {
        "instruction": (
            "Is this review positive, "
            "negative, or mixed: 'Best "
            "tacos in town, hands down. "
            "Parking is a nightmare "
            "though.'"
        ),
        "response": "Mixed.",
    },
    {
        "instruction": (
            "Suggest a one-line reply a "
            "restaurant owner could post "
            "in response to this review: "
            "'Waited an hour for a table "
            "even with a reservation.'"
        ),
        "response": (
            "We're sorry for the wait — "
            "we're working on improving "
            "our reservation system."
        ),
    },
    {
        "instruction": (
            "Summarize this review in one "
            "sentence: 'Portions were "
            "small for the price, but "
            "everything was fresh and "
            "beautifully plated.'"
        ),
        "response": (
            "Fresh, well-presented food, "
            "though portions run small "
            "for the price."
        ),
    },
])

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"],
        },
        {
            "role": "assistant",
            "content": example["response"],
        },
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False
        )
    }


formatted_dataset = raw_dataset.map(
    format_example
)

In [ ]:
# =========================================
# 2. Build and format the training data
# =========================================
from datasets import Dataset

# a small, concrete example dataset —
# replace with your real dataset
raw_dataset = Dataset.from_list([
    {
        "instruction": (
            "Summarize this review in one "
            "sentence: 'The pho was rich "
            "and flavorful, but we waited "
            "45 minutes just to order.'"
        ),
        "response": (
            "Great pho, but painfully "
            "slow service."
        ),
    },
    {
        "instruction": (
            "What star rating (1-5) fits "
            "this review: 'Cold food, "
            "rude staff, would not "
            "return.'"
        ),
        "response": "1 star.",
    },
    {
        "instruction": (
            "Extract the main complaint "
            "from this review: 'Loved "
            "the ambiance and cocktails, "
            "but the music was way too "
            "loud to hold a conversation.'"
        ),
        "response": "The music was too loud.",
    },
    {
        "instruction": (
            "Is this review positive, "
            "negative, or mixed: 'Best "
            "tacos in town, hands down. "
            "Parking is a nightmare "
            "though.'"
        ),
        "response": "Mixed.",
    },
    {
        "instruction": (
            "Suggest a one-line reply a "
            "restaurant owner could post "
            "in response to this review: "
            "'Waited an hour for a table "
            "even with a reservation.'"
        ),
        "response": (
            "We're sorry for the wait — "
            "we're working on improving "
            "our reservation system."
        ),
    },
    {
        "instruction": (
            "Summarize this review in one "
            "sentence: 'Portions were "
            "small for the price, but "
            "everything was fresh and "
            "beautifully plated.'"
        ),
        "response": (
            "Fresh, well-presented food, "
            "though portions run small "
            "for the price."
        ),
    },
])

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"],
        },
        {
            "role": "assistant",
            "content": example["response"],
        },
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False
        )
    }


formatted_dataset = raw_dataset.map(
    format_example
)

In [ ]:
# =========================================
# 3. Tokenize and split the data
# =========================================
def tokenize_example(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_example,
    remove_columns=(
        formatted_dataset.column_names
    ),
)

# split into train and validation sets
split_dataset = (
    tokenized_dataset.train_test_split(
        test_size=0.1
    )
)
tokenized_train = split_dataset["train"]
tokenized_val = split_dataset["test"]

In [ ]:
# =========================================
# 4. Data collator
# =========================================
from transformers import (
    DataCollatorForLanguageModeling,
)

# Qwen doesn't set a pad token by
# default; reuse the EOS token instead
if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

data_collator = (
    DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        # mlm=False tells the collator
        # this is causal (next-token)
        # LM, not masked LM like BERT
    )
)

In [ ]:
# =========================================
# 5. Training arguments and Trainer
# =========================================
from transformers import (
    TrainingArguments,
    Trainer,
)

# for generative models, eval loss
# (perplexity) is the usual metric,
# not accuracy or F1
training_args = TrainingArguments(
    output_dir="./lora-qwen-output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

In [ ]:
# =========================================
# 6. Test the model before training
# =========================================
import torch


def generate_reply(model, text):
    messages = [
        {"role": "user", "content": text},
    ]
    prompt = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )
    inputs = tokenizer(
        prompt, return_tensors="pt"
    )
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
        )
    reply_ids = outputs[
        0, inputs["input_ids"].shape[1]:
    ]
    return tokenizer.decode(
        reply_ids, skip_special_tokens=True
    )


test_review = (
    "Suggest a one-line reply a "
    "restaurant owner could post "
    "in response to this review: "
    "'The soup was cold and we "
    "waited 20 minutes for a table "
    "we'd already reserved.'"
)

# before training: the LoRA adapter
# is freshly initialized and has
# learned nothing yet
print("Before training:")
print(generate_reply(model, test_review))


In [ ]:
# =========================================
# 7. Train
# =========================================
trainer.train()


In [ ]:
# =========================================
# 8. Test the model after training
# =========================================
# after training: same model object,
# same prompt, but the adapter
# weights have now been updated
print("After training:")
print(generate_reply(model, test_review))


In [ ]:
# =========================================
# 9. Save the adapter
# =========================================
model.save_pretrained(
    "./my-qwen-lora-adapter"
    # typically a few MB, not GBs
)